# Notebook 16b — Balanced Metrics Patch

**Purpose:** Re-run the same CV as notebook 14 but additionally report:

- balanced_accuracy
- sensitivity (= recall)
- specificity = TN / (TN + FP)

**Output:** `results/tables/cv_full_metrics_with_balanced.csv`


In [1]:
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, balanced_accuracy_score, confusion_matrix
)
os.makedirs("../results/tables", exist_ok=True)
print("All imports OK")

All imports OK


In [2]:
df = pd.read_csv("../data/processed/windows_features_engineered.csv")
df = df.rename(columns={"Subject": "subject_id", "Stress_State": "label"})

FEATURE_COLS_FULL = [
    "Mean_RR", "SDNN", "RMSSD", "Mean_HR",
    "Resp_Rate", "Resp_Variability",
    "HRV_HR_Ratio", "Resp_Regularity", "Autonomic_Index"
]

print(f"Shape: {df.shape}")
print(f"Features: {len(FEATURE_COLS_FULL)}")

Shape: (599, 13)
Features: 9


In [3]:
def run_cv(df, feature_cols, modality_name):
    """
    Runs subject-wise 5-fold CV for 3 models on given feature_cols.
    Returns a summary dict: {model_name: {metric: (mean, std)}}
    """
    X_full   = df[feature_cols].values
    y_full   = df["label"].values
    subjects = df["subject_id"].values
    unique_subjects = np.unique(subjects)

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
        "SVM (RBF Kernel)"   : SVC(kernel="rbf", probability=True, random_state=42),
        "Random Forest"      : RandomForestClassifier(n_estimators=300, random_state=42),
    }

    fold_metrics = {
        name: {"accuracy":[], "precision":[], "recall":[], "f1":[],
               "roc_auc":[], "balanced_accuracy":[], "sensitivity":[], "specificity":[]}
        for name in models
    }

    for fold_idx, (train_idx, test_idx) in enumerate(kf.split(unique_subjects), 1):
        train_subjects = unique_subjects[train_idx]
        test_subjects  = unique_subjects[test_idx]

        train_mask = np.isin(subjects, train_subjects)
        test_mask  = np.isin(subjects, test_subjects)

        X_train, y_train = X_full[train_mask], y_full[train_mask]
        X_test,  y_test  = X_full[test_mask],  y_full[test_mask]

        scaler  = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test  = scaler.transform(X_test)

        for name, model in models.items():
            model_fresh = type(model)(**model.get_params())
            model_fresh.fit(X_train, y_train)
            y_pred = model_fresh.predict(X_test)
            y_prob = model_fresh.predict_proba(X_test)[:, 1]

            acc   = accuracy_score(y_test, y_pred)
            prec  = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
            rec   = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
            f1    = f1_score(y_test, y_pred, pos_label=1, zero_division=0)
            try:
                roc = roc_auc_score(y_test, y_prob)
            except ValueError:
                roc = float("nan")
            bal_acc = balanced_accuracy_score(y_test, y_pred)

            cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
            tn, fp = cm[0, 0], cm[0, 1]
            spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

            fold_metrics[name]["accuracy"].append(acc)
            fold_metrics[name]["precision"].append(prec)
            fold_metrics[name]["recall"].append(rec)
            fold_metrics[name]["f1"].append(f1)
            fold_metrics[name]["roc_auc"].append(roc)
            fold_metrics[name]["balanced_accuracy"].append(bal_acc)
            fold_metrics[name]["sensitivity"].append(rec)  # sensitivity = recall
            fold_metrics[name]["specificity"].append(spec)

    summary = {}
    for name in models:
        summary[name] = {}
        for metric, vals in fold_metrics[name].items():
            arr = np.array(vals)
            summary[name][metric] = (round(np.nanmean(arr), 4), round(np.nanstd(arr), 4))

    print(f"\n=== {modality_name} ===")
    for name in models:
        f1_m, f1_s = summary[name]["f1"]
        roc_m, roc_s = summary[name]["roc_auc"]
        print(f"  {name}: F1={f1_m:.3f}\u00b1{f1_s:.3f}  AUC={roc_m:.3f}\u00b1{roc_s:.3f}")

    return summary

print("run_cv() defined")

run_cv() defined


In [4]:
results = run_cv(df, FEATURE_COLS_FULL, "Multimodal (Full Features)")


=== Multimodal (Full Features) ===
  Logistic Regression: F1=0.583±0.131  AUC=0.834±0.076
  SVM (RBF Kernel): F1=0.589±0.206  AUC=0.836±0.103
  Random Forest: F1=0.576±0.194  AUC=0.818±0.109


In [5]:
# Build full table with all 8 metrics for all 3 models
METRICS = ["accuracy", "precision", "recall", "f1", "roc_auc", "balanced_accuracy", "sensitivity", "specificity"]
METRIC_LABELS = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC", "Balanced Accuracy", "Sensitivity", "Specificity"]

model_names = ["Logistic Regression", "SVM (RBF Kernel)", "Random Forest"]

rows = []
for name in model_names:
    row = {"Model": name}
    for metric, label in zip(METRICS, METRIC_LABELS):
        m, s = results[name][metric]
        row[label] = f"{m:.4f}\u00b1{s:.4f}"
    rows.append(row)

full_df = pd.DataFrame(rows)
full_df.to_csv("../results/tables/cv_full_metrics_with_balanced.csv", index=False)
print("Saved: results/tables/cv_full_metrics_with_balanced.csv\n")
print("Full Metrics Table (mean\u00b1std):")
print("=" * 120)
full_df.set_index("Model")

Saved: results/tables/cv_full_metrics_with_balanced.csv

Full Metrics Table (mean±std):


,Accuracy,Precision,Recall,F1,ROC-AUC,Balanced Accuracy,Sensitivity,Specificity
Model,,,,,,,,
Logistic Regression,0.7627±0.0681,0.6301±0.1063,0.5748±0.1930,0.5830±0.1310,0.8336±0.0764,0.7096±0.0932,0.5748±0.1930,0.8443±0.0800
SVM (RBF Kernel),0.7875±0.0707,0.6535±0.0851,0.5805±0.2696,0.5885±0.2063,0.8355±0.1033,0.7291±0.1250,0.5805±0.2696,0.8778±0.0424
Random Forest,0.7609±0.0876,0.5948±0.1412,0.5854±0.2423,0.5764±0.1944,0.8175±0.1086,0.7113±0.1240,0.5854±0.2423,0.8372±0.0795
